In [1]:
# Bull/Bear Market Regime Analysis - reloading locked Test-period results
# No new data pulled, no recomputation of DTW/OCP/TOP/naive 
# this notebook only reslices and re-analyzes returns already computed and saved

import pandas as pd
import numpy as np
import pickle
import math
from statsmodels.stats.multitest import multipletests

RISK_FREE_RATE = 0.045
TEST_START = '2018-01-01'
TEST_END = '2025-12-31'

# Loading the exact, locked Test-period return series from the original notebook
with open('test_backtest_results_sp500_20y.pkl','rb') as f:
    test_method_portfolio_results = pickle.load(f)

with open('naive_buyhold_returns_sp500_20y.pkl', 'rb') as f:
    naive_data = pickle.load(f)
    benchmark_returns = naive_data['benchmark_returns']

naive_test_returns = benchmark_returns.loc[TEST_START:TEST_END]

print('Loaded strategies:', list(test_method_portfolio_results.keys()))
for method in test_method_portfolio_results:
    r = test_method_portfolio_results[method]['returns']
    print(f'  {method}: {len(r)} weeks, {r.index[0].date()} to {r.index[-1].date()}')

print(f'\nNaive Buy-Hold: {len(naive_test_returns)} weeks, '
      f'{naive_test_returns.index[0].date()} to {naive_test_returns.index[-1].date()}')

Loaded strategies: ['DTW', 'OCP', 'TOP']
  DTW: 418 weeks, 2018-01-03 to 2025-12-31
  OCP: 418 weeks, 2018-01-03 to 2025-12-31
  TOP: 418 weeks, 2018-01-03 to 2025-12-31

Naive Buy-Hold: 418 weeks, 2018-01-03 to 2025-12-31


In [14]:
# Defining bear-market window from independently published market history

BEAR_WINDOWS = [
    ('2018-10-01', '2018-12-31'),  # Q4 2018 correction
    ('2020-02-19', '2020-03-23'),  # COVID crash
    ('2022-01-03', '2022-10-12'),  # 2022 bear market
]

def label_regime(index):
    """
    Labels each date as 'bear' if it falls within any of the defined
    bear-market windows, otherwise 'bull'. Everything not explicitly
    flagged as bear defaults to bull. This is a bifurcation, not an
    attempt to also isolate a separate "normal" middle regime.
    """
    labels = pd.Series('bull', index=index)
    for start, end in BEAR_WINDOWS:
        labels[(index >= start) & (index <= end)] = 'bear'
    return labels

all_series = {
    'DTW': test_method_portfolio_results['DTW']['returns'],
    'OCP': test_method_portfolio_results['OCP']['returns'],
    'TOP': test_method_portfolio_results['TOP']['returns'],
    'Naive': naive_test_returns
}

regime_labels = label_regime(all_series['DTW'].index)
n_bear = (regime_labels == 'bear').sum()
n_bull = (regime_labels == 'bull').sum()
print(f'Regime split: {n_bear} bear-market weeks, {n_bull} bull-market weeks (of {len(regime_labels)} total)')

regime_returns = {'bear': {}, 'bull': {}}
for name, series in all_series.items():
    regime_returns['bear'][name] = series[regime_labels == 'bear']
    regime_returns['bull'][name] = series[regime_labels == 'bull']

print('\nBear-market week counts by strategy (should all match):')
for name in all_series:
    print(f"  {name}: {len(regime_returns['bear'][name])} bear weeks, {len(regime_returns['bull'][name])} bull weeks")

Regime split: 59 bear-market weeks, 359 bull-market weeks (of 418 total)

Bear-market week counts by strategy (should all match):
  DTW: 59 bear weeks, 359 bull weeks
  OCP: 59 bear weeks, 359 bull weeks
  TOP: 59 bear weeks, 359 bull weeks
  Naive: 59 bear weeks, 359 bull weeks


In [20]:
# Metrics functions (copy and pasted from previous notebook)
def compute_backtest_metrics(returns_series, rf_annual=RISK_FREE_RATE):
    """
    Computes standard backtest metrics from a weekly return series.
    """
    returns_series = returns_series.dropna()

    cumulative = returns_series.cumsum()
    total_return = cumulative.iloc[-1] if len(cumulative) > 0 else 0.0

    weekly_rf = rf_annual / 52
    excess_returns = returns_series - weekly_rf
    sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(52) if excess_returns.std() > 0 else 0.0

    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    max_drawdown = drawdown.min()

    n_trades = (returns_series != 0).sum()
    win_rate = (returns_series > 0).sum() / n_trades if n_trades > 0 else 0

    return {
        'total_return': total_return,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'n_active_weeks': n_trades
    }
    
# Same Sortino and Calmar backtest as before
def compute_sortino_calmar(returns_series, rf_annual=RISK_FREE_RATE):
    returns_series = returns_series.dropna()
    n_periods = len(returns_series)

    weekly_rf = rf_annual / 52
    excess_returns = returns_series - weekly_rf

    # Sortino: only penalizes downside volatility.
    # Using all periods in the denominator
    downside_returns = excess_returns.clip(upper=0)
    downside_deviation = np.sqrt((downside_returns ** 2).mean())
    sortino = (excess_returns.mean() / downside_deviation) * np.sqrt(52) if downside_deviation > 0 else 0.0

    # Calmar: annualized return  over max drawdown
    cumulative = returns_series.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    max_drawdown = drawdown.min()

    total_return = cumulative.iloc[-1] if n_periods > 0 else 0.0
    annualized_return = total_return * (52 / n_periods) if n_periods > 0 else 0.0
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0.0

    return {'sortino': sortino, 'calmar': calmar}

# Computing metrics for every strategy, within each regime separately
regime_metrics = []

for regime in ['bear', 'bull']:
    for name in all_series:
        returns = regime_returns[regime][name]
        m = compute_backtest_metrics(returns)
        sc = compute_sortino_calmar(returns)

        regime_metrics.append({
            'regime': regime, 'strategy': name,
            'sharpe': m['sharpe'], 'sortino': sc['sortino'], 'calmar': sc['calmar'],
            'total_return': m['total_return'], 'max_drawdown': m['max_drawdown'],
            'n_weeks': len(returns)
        })

regime_metrics_df = pd.DataFrame(regime_metrics)
print('Regime-Conditional Performance (Bear vs. Bull, Test Period 2018-2025):')
print(regime_metrics_df[['regime', 'strategy', 'sharpe', 'sortino', 'calmar', 'max_drawdown', 'n_weeks']])

regime_metrics_df.to_csv('regime_conditional_metrics_sp500_20y.csv', index=False)
print('\nSaved: regime_conditional_metrics_sp500_20y.csv')

Regime-Conditional Performance (Bear vs. Bull, Test Period 2018-2025):
  regime strategy    sharpe   sortino    calmar  max_drawdown  n_weeks
0   bear      DTW  0.115644  0.163030  0.512192     -0.106727       59
1   bear      OCP -0.197215 -0.271404  0.299210     -0.092168       59
2   bear      TOP -1.659122 -1.877253 -0.566972     -0.234215       59
3   bear    Naive -2.611239 -2.742244 -0.891920     -0.817354       59
4   bull      DTW  0.891890  1.508595  1.391063     -0.085920      359
5   bull      OCP  0.818632  1.252519  1.222840     -0.108657      359
6   bull      TOP  0.579764  0.882749  0.787215     -0.138609      359
7   bull    Naive  1.369305  2.246201  1.372312     -0.186539      359

Saved: regime_conditional_metrics_sp500_20y.csv


In [25]:
# Significance testing, run separately with bull and bear markets

def block_bootstrap_sharpe_diff(returns_a, returns_b, rf_annual=RISK_FREE_RATE,
                                n_boot=5000, block_size=8, seed=42):
    rng = np.random.default_rng(seed)

    df = pd.concat([returns_a, returns_b], axis=1, join='inner').dropna()
    df.columns = ['a','b']
    n = len(df)
    weekly_rf = rf_annual / 52

    def sharpe(x):
        excess = x - weekly_rf
        return (excess.mean() / excess.std()) * np.sqrt(52) if excess.std() > 0 else 0

    observed_diff = sharpe(df['a']) - sharpe(df['b'])

    n_blocks = int(np.ceil(n / block_size))
    boot_diffs = np.empty(n_boot)

    for i in range(n_boot):
        starts = rng.integers(0, max(n - block_size + 1, 1), size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block_size) for s in starts])
        idx = idx[idx < n][:n]
        sample = df.iloc[idx]
        boot_diffs[i] = sharpe(sample['a']) - sharpe(sample['b'])

    p_value = 2 * min((boot_diffs >= 0).mean(), (boot_diffs <= 0).mean())
    ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])

    return {'observed_diff': observed_diff, 'p_value': p_value,
            'ci_lower': ci_lower, 'ci_upper': ci_upper}

comparisons = [
    ('DTW', 'Naive'), ('OCP', 'Naive'), ('TOP', 'Naive'),
    ('DTW', 'OCP'), ('DTW', 'TOP'), ('OCP', 'TOP')
]

all_regime_sig_results = []

for regime in ['bear', 'bull']:
    print(f'\n---{regime.upper()} MARKET ---')
    regime_sig_rows = []

    for a, b in comparisons:
        result = block_bootstrap_sharpe_diff(regime_returns[regime][a], regime_returns[regime][b])
        regime_sig_rows.append({
            'regime': regime, 'comparison': f'{a} vs {b}',
            'sharpe_diff': result['observed_diff'],
            'ci_95_lower': result['ci_lower'], 'ci_95_upper': result['ci_upper'],
            'p_value': result['p_value']
        })
        print(f"{a} vs {b}: diff={result['observed_diff']:.4f}, "
              f"95% CI=[{result['ci_lower']:.4f}, {result['ci_upper']:.4f}], p={result['p_value']:.4f}")

    regime_df = pd.DataFrame(regime_sig_rows)
    rejected, pvals_bh, _, _ = multipletests(regime_df['p_value'], alpha=0.05, method='fdr_bh')
    regime_df['p_value_bh_corrected'] = pvals_bh
    regime_df['significant_bh'] = rejected

    all_regime_sig_results.append(regime_df)

full_sig_df = pd.concat(all_regime_sig_results, ignore_index=True)
print('\n\nFull Regime-Conditional Significance Summary (BH-corrected within each regime):')
print(full_sig_df[['regime', 'comparison', 'sharpe_diff', 'p_value', 'p_value_bh_corrected', 'significant_bh']])

full_sig_df.to_csv('regime_significance_test_sp500_20y.csv', index=False)
print('\nSaved: regime_significance_test_sp500_20y.csv')


---BEAR MARKET ---
DTW vs Naive: diff=2.7269, 95% CI=[0.9342, 4.4190], p=0.0020
OCP vs Naive: diff=2.4140, 95% CI=[1.2510, 3.5252], p=0.0000
TOP vs Naive: diff=0.9521, 95% CI=[-0.3060, 2.0697], p=0.1680
DTW vs OCP: diff=0.3129, 95% CI=[-1.3514, 2.1073], p=0.7576
DTW vs TOP: diff=1.7748, 95% CI=[0.4753, 3.1863], p=0.0080
OCP vs TOP: diff=1.4619, 95% CI=[0.4673, 2.4662], p=0.0080

---BULL MARKET ---
DTW vs Naive: diff=-0.4774, 95% CI=[-1.3965, 0.4947], p=0.3348
OCP vs Naive: diff=-0.5507, 95% CI=[-1.4728, 0.5213], p=0.2868
TOP vs Naive: diff=-0.7895, 95% CI=[-1.7432, 0.2478], p=0.1300
DTW vs OCP: diff=0.0733, 95% CI=[-0.7108, 0.7576], p=0.8000
DTW vs TOP: diff=0.3121, 95% CI=[-0.3097, 0.9421], p=0.3024
OCP vs TOP: diff=0.2389, 95% CI=[-0.1831, 0.7716], p=0.2636


Full Regime-Conditional Significance Summary (BH-corrected within each regime):
   regime    comparison  sharpe_diff  p_value  p_value_bh_corrected  \
0    bear  DTW vs Naive     2.726883   0.0020               0.00600   
1    